<a href="https://colab.research.google.com/github/anushapatodia07/Jumbotail_assignment/blob/main/Assignment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Logic for calculations

Replenishment Logic

For each SKU, the goal is to estimate how much inventory is required, compare it with what is already available, and generate an order quantity while respecting operational constraints.


1.Demand Check
If there is no demand, no replenishment is needed.
max_drr = 0
final_suggestion = 0

2.Required Inventory
Inventory is planned based on expected demand and a safety buffer.
target_inventory = max_drr x inv_norm
safety_units = max_drr x safety_stock

required_inventory = target_inventory + safety_units


3. Available Inventory
Both current stock and incoming purchase orders are considered.
available_inventory = current_inventory + orderedquantity


4. Net Requirement
Order only if there is a shortage.
suggestion = required_inventory - available_inventory
If suggestion is negative, it is set to zero.


5. Case Constraint
Orders must be placed in full cases.
suggestion = ceil(suggestion / case_size) x case_size


6. Space Constraint
The order quantity cannot exceed available storage space.
available_space = max_allocated_space - available_inventory
final_suggestion = minimum of suggestion and available_space


7. Final Inventory
Inventory after placing the order:
final_inventory = available_inventory + final_suggestion


8. Days of Inventory
This shows how long inventory will last.
final_days_of_inventory = final_inventory / max_drr


9. Order Metrics
final_cases_suggestion = final_suggestion / case_size
final_value = final_suggestion x cp
final_tonnage = final_suggestion x deadweight


10. MOV Check
For vendors with value-based minimum order:
mov_check = 1 if final_value is greater than or equal to current_vendor_mov
otherwise mov_check = 0

Note: Results are stored in the output.csv file


In [2]:
import pandas as pd
import numpy as np
import json
from math import ceil

# Load data
df = pd.read_csv("assignment_data.csv")

# Fill nulls
df['current_inventory'] = df['current_inventory'].fillna(0)
df['orderedquantity'] = df['orderedquantity'].fillna(0)
df['max_drr'] = df['max_drr'].fillna(0)

# Function to compute suggestion
def compute_row(row):

    drr = row['max_drr']

    # Avoid divide-by-zero
    if drr == 0:
        return pd.Series([0, 0, 0, 0, 0, 0])

    # Step 1: Inventory targets
    target_inventory = drr * row['inv_norm']
    safety_units = drr * row['safety_stock']
    required_inventory = target_inventory + safety_units

    # Step 2: Available inventory
    available_inventory = row['current_inventory'] + row['orderedquantity']

    # Step 3: Raw suggestion
    suggestion = max(required_inventory - available_inventory, 0)

    # Step 4: Case rounding
    case_size = row['case_size']
    if case_size > 0:
        suggestion = ceil(suggestion / case_size) * case_size

    # Step 5: Space constraint
    max_space = row['max_allocated_space']
    available_space = max_space - available_inventory

    suggestion = min(suggestion, max(available_space, 0))

    # Step 6: Final calculations
    final_inventory = available_inventory + suggestion
    days_inventory = final_inventory / drr if drr > 0 else 0

    cases = suggestion / case_size if case_size > 0 else 0
    value = suggestion * row['cp']
    tonnage = suggestion * row['deadweight']

    # Step 7: MOV check
    mov_check = 0
    if row['minimum_order_criteria'] == "VALUE":
        mov_check = 1 if value >= row['current_vendor_mov'] else 0

    return pd.Series([
        int(suggestion),
        int(days_inventory),
        int(cases),
        int(value),
        tonnage,
        mov_check
    ])

# Apply function
df[[
    'final_suggestion',
    'final_days_of_inventory',
    'final_cases_suggestion',
    'final_value',
    'final_tonnage',
    'mov_check'
]] = df.apply(compute_row, axis=1)

# Save output
df.to_csv("output.csv", index=False)

## SQL Part

In [3]:
from google.colab import files
uploaded = files.upload()

Saving assignment_data.csv to assignment_data (1).csv


In [4]:
import sqlite3
import pandas as pd

# Load CSV
df = pd.read_csv("output.csv")

# Create DB
conn = sqlite3.connect("replenishment.db")

# Load into SQL table
df.to_sql("replenishment_data", conn, if_exists="replace", index=False)

1389

## Query - 1 Vendor Summary

In [5]:
query = """
SELECT
    vendor_name,
    SUM(final_value) AS total_suggested_value,
    SUM(final_cases_suggestion) AS total_cases
FROM replenishment_data
GROUP BY vendor_name;
"""

import pandas as pd
pd.read_sql(query, conn)

,vendor_name,total_suggested_value,total_cases
0,ANMOL INDUSTRIES LIMITED - Region Zeta,1553.0,2.0
1,BAJRANG TRADERS - Region Alpha,22000.0,3.0
2,BRITANNIA INDUSTRIES LIMITED - Region Epsilon,13600.0,20.0
3,COLGATE PALMOLIVE INDIA LTD - Region Beta,59283.0,6.0
4,DABUR INDIA LIMITED - Region Epsilon,7212.0,0.0
...,...,...,...
66,UNIBIC FOODS INDIA PRIVATE LIMITED - Region Delta,3502.0,1.0
67,VARUN BEVERAGES LIMITED - Region Beta,0.0,0.0
68,VARUN BEVERAGES LIMITED - Region Epsilon,0.0,0.0
69,VIBHAVA MARKETING CORPORATION OD,974.0,1.0


## Query 2 - Top 10 Riskiest SKUs

In [6]:
query2 = """
SELECT
    facility_name,
    vendor_name,
    title,
    current_inventory,
    max_drr,
    (current_inventory * 1.0 / max_drr) AS current_days_inventory
FROM replenishment_data
WHERE max_drr > 0
ORDER BY current_days_inventory ASC
LIMIT 10;
"""

pd.read_sql(query2, conn)

,facility_name,vendor_name,title,current_inventory,max_drr,current_days_inventory
0,FC-Alpha,DABUR INDIA LIMITED - Region Epsilon,"Real Koolerz Fruit Drink, Mango, 10Rs Tetra Pak",0,118,0.0
1,FC-Alpha,SIDDHI VINAYAK TRADERS - Region Alpha,"Cipla Prolyte ORS, Mixed Fruit, 200ml Tetra Pak",0,4,0.0
2,FC-Alpha,SIDDHI VINAYAK TRADERS - Region Alpha,"Cipla Prolyte ORS, Nimbu Paani, 200ml Tetra Pak",0,6,0.0
3,FC-Alpha,SIDDHI VINAYAK TRADERS - Region Alpha,"Cipla Prolyte ORS, Apple, 200ml Tetra Pak",0,50,0.0
4,FC-Alpha,SIDDHI VINAYAK TRADERS - Region Alpha,"Campa Soft Drink, Orange, 500ml Bottle",0,72,0.0
5,FC-Alpha,SIDDHI VINAYAK TRADERS - Region Alpha,"Campa Soft Drink, Cola, 200ml Bottle",0,276,0.0
6,FC-Alpha,Shree Krishna Enterprises - Region Epsilon,"Campa Soft Drink, Cola, 200ml Bottle",0,276,0.0
7,FC-Alpha,SIDDHI VINAYAK TRADERS - Region Alpha,"Campa Soft Drink, Orange, 200ml Bottle",0,144,0.0
8,FC-Alpha,Shree Krishna Enterprises - Region Epsilon,"Campa Soft Drink, Orange, 200ml Bottle",0,144,0.0
9,FC-Alpha,SLMG BEVERAGES PRIVATE LIMITED - Region Alpha,"Limca Soft Drink, 2L Bottle",0,43,0.0


## Query 3 - Zero Demand SKUs

In [9]:
query10 = """
SELECT
    COUNT(*) AS zero_demand_skus
FROM replenishment_data
WHERE max_drr = 0;
"""

pd.read_sql(query10, conn)

,zero_demand_skus
0,270


## Query - 4 Procurement by Category

In [10]:
query8 = """
SELECT
    category_name,
    SUM(final_value) AS total_procurement_value
FROM replenishment_data
GROUP BY category_name
ORDER BY total_procurement_value DESC;
"""

pd.read_sql(query8, conn)

,category_name,total_procurement_value
0,"Biscuit, Confectionary & Snacks",561599.0
1,Personal Care,475896.0
2,Home Care,235665.0
3,Grounded Spices & Masalas,157463.0
4,Beverages,150969.0
5,Packaged Foods,81220.0
6,Dairy,60270.0
7,Salt,20459.0
8,Office & Stationary Products,19308.0
9,Tea and Coffee,17046.0


## Query 5 - Total Procurement Summary

In [11]:
query9 = """
SELECT
    SUM(final_value) AS total_value,
    SUM(final_cases_suggestion) AS total_cases,
    SUM(final_suggestion) AS total_units
FROM replenishment_data;
"""

pd.read_sql(query9, conn)

,total_value,total_cases,total_units
0,1811535.0,1004.0,150870.0
